[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Object-Oriented Python](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)

# Properties &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

Run the cell below first.


In [1]:
import functools

print("ready")


ready


**1.** A computed property that stays current.


In [2]:
class Station:
    def __init__(self, readings):
        self.readings = readings

    @property
    def spread(self):
        return round(max(self.readings) - min(self.readings), 1)


north = Station([-4.1, -2.6])
print("spread:", north.spread)

north.readings.append(-9.9)
print("after a colder reading:", north.spread)


spread: 1.5
after a colder reading: 7.3


`spread` is read like data and calculated like a method. It never needs updating, because it is
worked out from `readings` at the moment it is read.


**2.** A checked, tidied `name`.


In [3]:
class Station:
    def __init__(self, name):
        self.name = name

    @property
    def name(self):
        return self._name

    @name.setter
    def name(self, value):
        value = value.strip()
        if not value:
            raise ValueError("a station needs a name")
        self._name = value


north = Station("  Tromso  ")
print(repr(north.name))

try:
    north.name = "   "
except ValueError as error:
    print("rejected:", error)
print("still:", repr(north.name))


'Tromso'
rejected: a station needs a name
still: 'Tromso'


The setter does two jobs: it tidies the value and then it checks it, in that order, so a name made
only of spaces is caught after the spaces are removed.

`__init__` assigns `self.name`, not `self._name`, so the station's first name goes through the same
check as every later one.


**3.** Assigning to a property with no setter.


In [4]:
class Station:
    def __init__(self, readings):
        self.readings = readings

    @property
    def spread(self):
        return round(max(self.readings) - min(self.readings), 1)


north = Station([-4.1, -2.6])

try:
    north.spread = 0
except AttributeError as error:
    print("AttributeError:", error)

# This failing is correct. spread is calculated from the readings, so a value
# assigned to it would disagree with them the moment it was stored. The only
# way to change the spread is to change the readings it is calculated from.


AttributeError: property 'spread' of 'Station' object has no setter


A property without a setter is read-only, and for a calculated value that is the design rather
than an accident.


**4.** One stored value, two views.


In [5]:
class Reading:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def fahrenheit(self):
        return round(self.celsius * 9 / 5 + 32, 1)

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = round((value - 32) * 5 / 9, 1)


boiling = Reading(0.0)
boiling.fahrenheit = 212

print("celsius:   ", boiling.celsius)
print("fahrenheit:", boiling.fahrenheit)
print("stored:    ", vars(boiling))


celsius:    100.0
fahrenheit: 212.0
stored:     {'celsius': 100.0}


Only `celsius` is stored. Setting `fahrenheit` converts and stores Celsius, and reading
`fahrenheit` converts back, so the two can never disagree.


**5.** A setter that checks every reading.


In [6]:
class Station:
    def __init__(self, name, readings):
        self.name = name
        self.readings = readings

    @property
    def readings(self):
        return self._readings

    @readings.setter
    def readings(self, values):
        bad = [v for v in values if not -90 <= v <= 60]
        if bad:
            raise ValueError(f"readings out of range: {bad}")
        self._readings = list(values)


north = Station("Tromso", [-4.1, -2.6])
print("accepted:", north.readings)

try:
    north.readings = [-4.1, 999.0]
except ValueError as error:
    print("rejected:", error)

try:
    Station("Bodo", [999.0])
except ValueError as error:
    print("rejected at construction:", error)

north.readings.append(999.0)
print("but append went around the setter:", north.readings)


accepted: [-4.1, -2.6]
rejected: readings out of range: [999.0]
rejected at construction: readings out of range: [999.0]
but append went around the setter: [-4.1, -2.6, 999.0]


The setter checks every item and reports the ones that failed, which is more useful than saying only
that something did. Because `__init__` assigns `self.readings`, a station cannot be created with an
impossible reading either.

The last line shows the limit of a setter. `append` changes the list that the getter handed back; it
never assigns to `readings`, so the setter never runs. A setter guards assignment to the attribute,
not changes made inside the value it holds.


**6.** A cached value, and when it goes stale.


In [7]:
class Archive:
    def __init__(self, readings):
        self.readings = readings

    @functools.cached_property
    def summary(self):
        print("  (working out the summary)")
        return {"n": len(self.readings), "coldest": min(self.readings)}


march = Archive([-4.1, -2.6])
print(march.summary)
print(march.summary)

march.readings.append(-9.9)
print("after a colder reading:", march.summary)

# The summary still says two readings with a coldest of -4.1. cached_property
# stores the first answer and never looks at readings again. That is fine for
# an archive that is complete when it is created and never changes. It is wrong
# for anything that keeps receiving readings.


  (working out the summary)
{'n': 2, 'coldest': -4.1}
{'n': 2, 'coldest': -4.1}
after a colder reading: {'n': 2, 'coldest': -4.1}


`working out the summary` printed once for three reads, and the third read is stale. Both are the
same fact: the value was computed on the first read and stored on the object.


---

&#8592; **Back to:** [Properties](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/object-oriented-python/07-properties.ipynb)  &nbsp;&middot;&nbsp;  [Object-Oriented Python Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/object-oriented-python.html)
